# 03 Pull Census ACS Data

## Purpose

This notebook pulls tract-level demographic data from the U.S. Census Bureau's American Community Survey (ACS) API for Harris County, Texas.

This step matters because the project needs population and income data to evaluate whether Houston METRO service aligns with areas of population density and potential transit need.

## Inputs

- Census ACS 5-year API
- Geography: Census tracts in Harris County, Texas
- Variables:
  - `B01003_001E`: Total population
  - `B19013_001E`: Median household income

## Outputs

- `data/processed/acs_harris_tracts.csv`

This processed CSV is later joined with Census tract geometry to create population-density maps.

## 1. Import libraries

`pandas` is used to structure the API response as a dataframe.  
`requests` is used to call the Census API.  
`os` is used so the API key can be stored safely as an environment variable instead of hard-coded in the notebook.

In [ ]:
import os
import pandas as pd
import requests

## 2. Set Census API request parameters

The Census API requires a key for this request. To avoid exposing the key in GitHub, store it as an environment variable named `CENSUS_API_KEY`.

If running locally, you can temporarily replace `os.getenv("CENSUS_API_KEY")` with your key, but do not commit the key to GitHub.

In [ ]:
CENSUS_API_KEY = os.getenv("CENSUS_API_KEY")

if CENSUS_API_KEY is None:
    raise ValueError(
        "CENSUS_API_KEY is not set. Set it as an environment variable before running this notebook."
    )

params = {
    "get": "NAME,B01003_001E,B19013_001E",
    "for": "tract:*",
    "in": "state:48 county:201",
    "key": CENSUS_API_KEY
}

## 3. Request ACS tract-level data

This request pulls all Census tracts in Harris County, Texas. The response should return a status code of `200` and a JSON-style table where the first row contains column names.

In [ ]:
response = requests.get(
    "https://api.census.gov/data/2023/acs/acs5",
    params=params,
    timeout=15
)

print("Status code:", response.status_code)
print(response.text[:500])

## 4. Convert the API response into a dataframe

The Census API returns data as a list of rows. The first row contains headers, so the notebook separates the header row from the data rows and converts the result into a Pandas dataframe.

In [ ]:
data = response.json()

acs = pd.DataFrame(
    data[1:],
    columns=data[0]
)

acs.head()

## 5. Create a Census tract GEOID

The TIGER/Line shapefile uses a single `GEOID` field to identify each tract. The ACS API returns state, county, and tract separately, so this step combines them into one matching field.

This join key is necessary for mapping demographic values onto tract boundaries.

In [ ]:
acs["GEOID"] = (
    acs["state"].astype(str).str.zfill(2)
    + acs["county"].astype(str).str.zfill(3)
    + acs["tract"].astype(str).str.zfill(6)
)

acs.head()

## 6. Rename variables for readability

The original Census variable names are useful for documentation, but cleaner names make later notebooks easier to read.

In [ ]:
acs_clean = acs.rename(
    columns={
        "B01003_001E": "population",
        "B19013_001E": "median_household_income"
    }
).copy()

acs_clean["population"] = pd.to_numeric(
    acs_clean["population"],
    errors="coerce"
)

acs_clean["median_household_income"] = pd.to_numeric(
    acs_clean["median_household_income"],
    errors="coerce"
)

acs_clean.head()

## 7. Inspect the cleaned ACS data

This summary check helps confirm that the population and income fields were converted correctly.

In [ ]:
acs_clean[
    ["population", "median_household_income"]
].describe()

## 8. Save processed ACS data

The cleaned dataset is saved to the `data/processed` folder so it can be reused by later notebooks without calling the Census API again.

In [ ]:
acs_clean.to_csv(
    "../data/processed/acs_harris_tracts.csv",
    index=False
)

print("Saved: ../data/processed/acs_harris_tracts.csv")

## 9. Verify saved file

This final check confirms that the processed CSV was written successfully.

In [ ]:
pd.read_csv(
    "../data/processed/acs_harris_tracts.csv"
).head()

## Summary

This notebook successfully pulls Harris County tract-level ACS data, creates a matching Census `GEOID`, renames key demographic variables, and saves a processed dataset for later mapping.

The resulting file supports the population-density and equity components of the Houston transit investment analysis.